### Scratchpad for Development

This notebook was used during development for quick testing and exploration of example cases related to the framework.

In [2]:
from src.solver import *
from src.train import *

In [ ]:
# Variabili della paziente P
age  = Int('age')
bmi  = Int('bmi')
ant   = Int('ant')
a1c  = Int('a1c')
hb   = Int('hb')

# Paziente generico Q
age2  = Int('age2')
bmi2  = Int('bmi2')
ant2   = Int('ant2')
a1c2  = Int('a1c2')
hb2   = Int('hb2')

s = Solver()

# Domini per P
s.add(age >= 0, age <= 2)
s.add(bmi >= 0, bmi <= 2)
s.add(ant >= 0, ant <= 2)
s.add(a1c >= 0, a1c <= 2)
s.add(hb >= 0, hb <= 2)

# Domini per Q
s.add(age2 >= 0, age2 <= 2)
s.add(bmi2 >= 0, bmi2 <= 2)
s.add(ant2 >= 0, ant2 <= 2)
s.add(a1c2 >= 0, a1c2 <= 2)
s.add(hb2 >= 0, hb2 <= 2)

# Classificatori
# M = 3*bmi + hb - 5
# P = -5*age -3 *bmi - hb + 5
D = age + bmi - 3
P = 0.5*ant + a1c - 2.5 - bmi
cD = - ant - a1c + ant2 + a1c2 - 1
cB = ant - ant2

M = 3*bmi + hb - 5
M2 = 3*bmi2 + hb2 - 5
A = age - 1
A2 = age2 - 1

#s.add(D>=0)
#s.add(P>=0)
s.add(cD>=0)
s.add(Or(M >= 0, M2 >= 0))
s.add(Or(M < 0, M2 < 0))
s.add(Or(A >= 0, A2 >= 0))
s.add(Or(A < 0, A2 < 0))
s.add(cB>=0)

#qui tutte le donne incinte diabetiche violano il vincolo di inclusione tra ruoli. quindi nella semantica widtio non esistono donne incinte diabetiche.

# Risoluzione
if s.check() == sat:
    m = s.model()
    print("Soluzione trovata per a:")
    print(f"age     = {m[age]}")
    print(f"bmi     = {m[bmi]}")
    print(f"ant     = {m[ant]}")
    print(f"a1c     = {m[a1c]}")
    print(f"hb      = {m[hb]}")
    print("Soluzione trovata per b:")
    print(f"age     = {m[age2]}")
    print(f"bmi     = {m[bmi2]}")
    print(f"ant     = {m[ant2]}")
    print(f"a1c     = {m[a1c2]}")
    print(f"hb      = {m[hb2]}")
else:
    print("Nessuna soluzione trovata.")

Soluzione trovata per a:
age     = 1
bmi     = 1
ant     = 0
a1c     = 0
hb      = 2
Soluzione trovata per b:
age     = 0
bmi     = 0
ant     = 1
a1c     = 0
hb      = 0


In [ ]:
from z3 import *
from itertools import product

# Variabili della paziente P
age, bmi, ant, a1c, hb = Ints('age bmi ant a1c hb')
s = Solver()

# Dominio ridotto per velocità
range_age  = range(0, 101)
range_bmi  = range(10, 51)
range_ant  = range(0, 3)
range_a1c  = range(3, 16)
range_hb   = range(5, 26)

# Domini per P
s.add(age >= 0, age <= 100)
s.add(bmi >= 10, bmi <= 50)
s.add(ant >= 0, ant <= 2)
s.add(a1c >= 3, a1c <= 15)
s.add(hb >= 5, hb <= 25)

# Classificatori per P
M = age/5 + bmi/3 - (3*hb)/4 - 2.5
D = age/4 - (2*bmi)/3 + (3*a1c)/2 - 0.25
# s.add(M < 0)  # donna
# s.add(D >= 0) # diabetica

# Verifica implicazioni per ogni combinazione di Q
for age2_val, bmi2_val, ant2_val, a1c2_val, hb2_val in product(range_age, range_bmi, range_ant, range_a1c, range_hb):
    CD      = (4*(age - age2_val))/5 + (a1c - a1c2_val)/4 - 2*(ant - ant2_val)/5 - 3/2
    CB      = (hb - hb2_val)/6 + (3*(ant - ant2_val))/4 - 1
    CD_back = (4*(age2_val - age))/5 + (a1c2_val - a1c)/4 - 2*(ant2_val - ant)/5 - 3/2
    CB_back = (hb2_val - hb)/6 + (3*(ant2_val - ant))/4 - 1

    s.add(Implies(CD >= 0, CB >= 0))
    s.add(Implies(CD_back >= 0, CB_back >= 0))

# Risoluzione
if s.check() == sat:
    m = s.model()
    print("Soluzione trovata:")
    print(f"age     = {m[age]}")
    print(f"bmi     = {m[bmi]}")
    print(f"ant     = {m[ant]}")
    print(f"a1c     = {m[a1c]}")
    print(f"hb      = {m[hb]}")
else:
    print("Nessuna soluzione trovata.")

In [ ]:
def D(p):  # Diabetes
    age, bmi, agcount, a1c, hb = p
    value = (1/4)*age - (2/3)*bmi + (3/2)*a1c - (1/4)
    return int(value >= 0)

def M(p):  # Male
    age, bmi, agcount, a1c, hb = p
    value = (1/3)*a1c + (1/5)*bmi - (3/4)*hb + (5/2)
    return int(value >= 0)

def P(p):  # Pregnant
    age, bmi, agcount, a1c, hb = p
    value = (1/3)*age - (2/5)*hb + (3/4)
    return int(value >= 0)

def CB(p1, p2):  # Compatible Blood
    _, _, ag1, _, _ = p1
    age2, _, _, _, _ = p2
    value = 50*ag1 - age2 - 1
    return int(value >= 0)

def CD(p1, p2):  # Compatible Donor
    age1, _, ag1, a1c1, _ = p1
    age2, _, ag2, a1c2, _ = p2
    value = (4/5)*(age1 - age2) + (1/4)*(a1c1 - a1c2) - (2/5)*(ag1 - ag2) - 3/2
    return int(value >= 0)

In [ ]:
e = [100, 20, 0, 6, 12] #never in compDon, female
print("Male(e)?:", M(e))

e1 = [30, 28, 1, 7, 10] #both male and pregnant
print("Male(e1)?:", M(e1), "Pregnant(e1)?:", P(e1))

# e_1 = [25, 19, 0, 6, 12] #both females, both in CD but not in CB
# e_2 = [22, 18, 2, 8, 20]
# print(P(e_1))
# print(M(e_1))
# print(D(e_1))
# print(P(e_2))
# print(M(e_2))
# print(D(e_2))
# print(CB(e_1, e_2))
# print(CD(e_1, e_2))

# e_3 = [45, 18, 1, 10, 20] #e3 female, diabetic, can donate blood to e4, wich is also female
# e_4 = [21, 18, 2, 5, 24]
# print(P(e_3))
# print(M(e_3))
# print(D(e_3))
# print(P(e_4))
# print(M(e_4))
# print(D(e_4))
# print(CB(e_3, e_4))


In [ ]:
e = [30, 28, 1, 7, 10]
print("Pregnant:", P(e), "Male:", M(e), "-->", e)

print("----------------------------")

e_1 = [25, 19, 0, 6, 12]
e_2 = [22, 18, 2, 8, 20]
print("Pregnant:", P(e_1), "Male:", M(e_1), "-->", e_1)
print("Pregnant:", P(e_2), "Male:", M(e_2), "-->", e_2)
print("CompatibleDonor:", CD(e_1, e_2), "CompatibleBlood:", CB(e_1, e_2), "-->", e_1, e_2)
print("CompatibleDonor:", CD(e_2, e_1), "CompatibleBlood:", CB(e_2, e_1), "-->", e_2, e_1)
# e = [6, 15, 0, 7, 5]
# print(D(e))
# print(M(e))

e = [25, 22, 0, 3, 5]
print(D(e))
print(M(e))
print(P(e))

e = [24, 18, 0, 5, 11]
print(D(e))
print(M(e))
print(P(e))